In [1]:
try:
    import math as my
except ImportError as e:
    print(f'{e}')
except ModuleNotFoundError as e:
    print(f'{e}')
    
try:
    import pandas as pd
    print(f'Pandas {pd.__version__} loaded successfully')
except ImportError as e:
    print(f'{e}')
except ModuleNotFoundError as e:
    print(f'{e}')
    
try:
    import numpy as np
    print(f'Numpy {np.__version__} loaded successfully')
except ImportError as e:
    print(f'{e}')
except ModuleNotFoundError as e:
    print(f'{e}')
    
try:
    import xarray as xr
    print(f'Xarray {xr.__version__} loaded successfully')
except ImportError as e:
    print(f'{e}')
except ModuleNotFoundError as e:
    print(f'{e}')
    
try:
    import matplotlib as mat
    import matplotlib.pyplot as plt
    print(f'Matplotlib {mat.__version__} loaded successfully')
except ImportError as e:
    print(f'{e}')
except ModuleNotFoundError as e:
    print(f'{e}')

try:
    import calendar as cally
except ImportError as e:
    print(f'{e}')
except ModuleNotFoundError as e:
    print(f'{e}')    

try:
    import time as time
except ImportError as e:
    print(f'{e}')
except ModuleNotFoundError as e:
    print(f'{e}')

try:
    import scipy as sp
except ImportError as e:
    print(f'{e}')
except ModuleNotFoundError as e:
    print(f'{e}')
    
try:
    from datetime import timedelta
except ImportError as e:
    print(f'{e}')
except ModuleNotFoundError as e:
    print(f'{e}')
    
try:
    import regionmask as remy
    print(f'Regionmask {remy.__version__} loaded successfully')
except ImportError as e:
    print(f'{e}')
except ModuleNotFoundError as e:
    print(f'{e}')

try :
    from netCDF4 import Dataset
    import h5py
except ImportError as e :
    print(f'{e}')
except ModuleNotFoundError as e :
    print(f'{e}')
try :
    import os
    os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
except Exception as e :
    print(f'{e}')

import sys

print(f' ########### \n    READY   \n ###########')

#HadGEM3-GC31-LL DIDNT WORK WITH COORDS- tasmax
#KACE-1-0-G   DIDNT WORK WITH COORDS
#UKESM1-0-LL DIDN'T WORK WITH COORDS

Pandas 2.3.0 loaded successfully
Numpy 2.2.6 loaded successfully
Xarray 2025.7.0 loaded successfully
Matplotlib 3.10.3 loaded successfully
Regionmask 0.13.0 loaded successfully
 ########### 
    READY   
 ###########


In [2]:
model = input(f'Which model is subsetting? ')
series = input(f'Which series are is this in? ')
var = input(f'Which variable? ')
loc = input(f"Where are we subsetting? (Currently set for 'NE' or 'USA')  ")

invalid = ('HadGEM3-GC31-LL', 'KACE-1-0-G', 'UKESM1-0-LL')#, 'ACCESS-CM2')



start_time = time.perf_counter()

if model in invalid:
    print(f'{model} is invalid for current subsetting \n Terminating')
    sys.exit()
else :
    pass

if series == 'Historical':
    subset = f'/mnt/nrdstor/dixon/mudby97/FWI_test/dap.ceda.ac.uk/badc/evoflood/data/Downscaled_CMIP6_Climate_Data/{var}/Historical/Global_{var}_Downscaled_{model}_1981-2014_compressed.nc'
else :
    subset =f'/mnt/nrdstor/dixon/mudby97/FWI_test/dap.ceda.ac.uk/badc/evoflood/data/Downscaled_CMIP6_Climate_Data/{var}/ssp585/Global_{var}_Downscaled_{model}_2015-2100_ssp585_compressed.nc'

if model == 'INM-CM4-8' or model == 'INM-CM5-0' :
    if loc == 'NE' :
        place = 'Nebraska'
        latidx = slice(39,43)
        lonidx = slice(256,265)
    elif loc == 'USA' :
        place = loc
        latidx = slice(23,49)
        lonidx = slice(235,293)
elif model not in invalid and model != 'INM-CM4-8' and model != 'INM-CM5-0' :
    if loc == 'NE' :
        place = 'Nebraska'
        latidx = slice(40,43)
        lonidx = slice(256,265)
    elif loc == 'USA' :
        place = loc
        latidx = slice(24,49)
        lonidx = slice(235,293)
else :
    print(f'Location not indexed. Terminating.')
    sys.exit
tocut = xr.open_dataset(subset)    
print(tocut.lat.values[:10])
print(tocut.lat.values[-10:])
consent = input(f'Are these valid coordinates? Should we proceed Y/N? ').upper()
if consent == 'N' or consent == 'NO' :
    print('Terminating')
    sys.exit()
else :
    print(f'Subsetting {model} {series} data for {var} for {loc} coordinates')

#existentialcrisis = os.path.exists(tocut)
#filesize = os.path.getsize(tocut)

#if existentialcrisis == 'True' and filesize >= 20 :
    #for engine in ("netcdf4", "h5netcdf"):
        #try :
            #print("trying", engine)
            #tocut = xr.open_dataset(subset, engine=engine)
            #print("opened with", engine)
        #except Exception as e:
           # print(engine, "failed:", type(e).__name__, e)
       # print(tocut.dims)
#else :
   # pass
    try :    
        cutfile = tocut[{var}].isel(time=slice(1,31411)).sel(
            lat=latidx,
            lon=lonidx)

        try :
            cutfile.to_netcdf(f'Downscaled_CMIP6_{var}_{place}_{series}_{model}.nc')
            print(f'+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+\n                Success!                       \nSaved {var} as an .nc of the {series} {model}\n+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+_+')              
        except Exception as eeee:
            print(f'{eeee}')

        try :
            tocut.close()
            print(f'{subset} closed successfully!')
        except Exception as eeeee:
            print(f'{eeeee}')

    except Exception as shucks : 
        print(f'Unable to subset.\n {shucks}\n Terminating')
        sys.exit()
    
    
loop_time = timedelta(seconds=time.perf_counter() - start_time)
print(f"This took us about {loop_time}")

Which model is subsetting? ACCESS-CM2
Which series are is this in? ssp585
Which variable? tasmax
Where are we subsetting? (Currently set for 'NE' or 'USA')  NE


OSError: [Errno -101] NetCDF: HDF error: '/mnt/nrdstor/dixon/mudby97/FWI_test/dap.ceda.ac.uk/badc/evoflood/data/Downscaled_CMIP6_Climate_Data/tasmax/ssp585/Global_tasmax_Downscaled_ACCESS-CM2_2015-2100_ssp585_compressed.nc'

In [5]:
testmodel = input(f'What model are we testing?' )
testmode = input(f'Over which series? ')
if testmode == 'Historical' :
    testnow = 'Full'
else :
    testnow = input(f"'Full' or 'NearFuture' or 'MidFuture' or 'FarFuture' ")
testinterp = input (f"'NE' or 'USA'? ").upper()
if testinterp == 'NE' :
    testlocale = 'Nebraska'
elif testinterp == 'USA' :
    testlocale = testinterp
else : 
    print('Error. Location not coded.')
    sys.exit()
if testmode == 'Historical' :
    try :
        temp = f'vardata/cmip6/{testmode}/tasmax/Downscaled_CMIP6_tasmax_{testlocale}_Historical_{testmodel}.nc'
        testtemp = xr.open_dataset(temp)
        print(f"{testtemp.coords} \n+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+\n{testtemp.values}")
        try:
            testtemp.close()
            print(f'{temp} closed successfully!')
        except Exception as ugh:
            print(f'{ugh}')
    except Exception as zoinks :
        print(f' Temperature NetCDF not available. \n {zoinks}')
    try :
        pr = f'vardata/cmip6/{testmode}/pr/Downscaled_CMIP6_pr_{testlocale}_Historical_{testmodel}.nc'
        testpr = xr.open_dataset(pr)
        print(f"{testpr.coords} \n+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+\n{testpr.values}")
        try:
            testpr.close()
            print(f'{pr} closed successfully!')
        except Exception as ugh:
            print(f'{ugh}')
    except Exception as jeepers :
        print(f' Precipitation NetCDF not available. \n {jeepers}')
    try :
        hurs = f'vardata/cmip6/{testmode}/hurs/Downscaled_CMIP6_hurs_{testlocale}_Historical_{testmodel}.nc'
        testhurs = xr.open_dataset(hurs)
        print(f"{testhurs.coords} \n+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+\n{testhurs.values}")
        try:
            testhurs.close()
            print(f'{hurs} closed successfully!')
        except Exception as ugh:
            print(f'{ugh}')
    except Exception as criminy :
        print(f' Relative Humidity NetCDF not available. \n {criminy}')
    try :
        sfcwind = f'vardata/cmip6/{testmode}/sfcwind/Downscaled_CMIP6_sfcWind_{testlocale}_Historical_{testmodel}.nc'
        testwind = xr.open_dataset(sfcWind)
        print(f"{testwind.coords} \n+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+\n{testtemp.values}")
        try:
            testwind.close()
            print(f'{sfcWind} closed successfully!')
        except Exception as ugh:
            print(f'{ugh}')
    except Exception as goodlordalmighty :
        print(f' Surface Wind NetCDF not available. \n {goodlordalmighty}')    
#ELIF FOR SSP585 HERE
elif testmode == 'ssp585' or testmode == 'SSP585' :
    testmode = 'ssp585'
    try :
        temp = f'vardata/cmip6/{testmode}/tasmax/Downscaled_CMIP6_tasmax_{testlocale}_ssp585_{testmodel}.nc'
        testtemp = xr.open_dataset(temp)
        print(f"{testtemp.coords} \n+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+\n{testtemp.values}")
        try:
            testtemp.close()
            print(f'{temp} closed successfully!')
        except Exception as ugh:
            print(f'{ugh}')
    except Exception as zoinks :
        print(f' Temperature NetCDF not available. \n {zoinks}')
    try :
        pr = f'vardata/cmip6/{testmode}/pr/Downscaled_CMIP6_pr_{testlocale}_ssp585_{testmodel}.nc'
        testpr = xr.open_dataset(pr)
        print(f"{testpr.coords} \n+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+\n{testpr.values}")
        try:
            testpr.close()
            print(f'{pr} closed successfully!')
        except Exception as ugh:
            print(f'{ugh}')
    except Exception as jeepers :
        print(f' Precipitation NetCDF not available. \n {jeepers}')
    try :
        hurs = f'vardata/cmip6/{testmode}/hurs/Downscaled_CMIP6_hurs_{testlocale}_ssp585_{testmodel}.nc'
        testhurs = xr.open_dataset(hurs)
        print(f"{testhurs.coords} \n+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+\n{testhurs.values}")
        try:
            testhurs.close()
            print(f'{hurs} closed successfully!')
        except Exception as ugh:
            print(f'{ugh}')
    except Exception as criminy :
        print(f' Relative Humidity NetCDF not available. \n {criminy}')
    try :
        sfcwind = f'vardata/cmip6/{testmode}/sfcwind/Downscaled_CMIP6_sfcWind_{testlocale}_ssp585_{testmodel}.nc'
        testwind = xr.open_dataset(sfcWind)
        print(f"{testwind.coords} \n+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+\n{testtemp.values}")
        try:
            testwind.close()
            print(f'{sfcWind} closed successfully!')
        except Exception as ugh:
            print(f'{ugh}')
    except Exception as goodlordalmighty :
        print(f' Surface Wind NetCDF not available. \n {goodlordalmighty}')
else :
    print(f'Nothing found.')

What model are we testing?INM-CM4-8
Over which series? ssp585
'Full' or 'NearFuture' or 'MidFuture' or 'FarFuture' Full
'NE' or 'USA'? NE
Coordinates:
  * lat      (lat) float64 96B 40.2 40.45 40.7 40.95 ... 42.2 42.45 42.7 42.95
  * lon      (lon) float64 288B 256.1 256.3 256.6 256.8 ... 264.3 264.6 264.8
  * time     (time) datetime64[ns] 251kB 2015-01-01 2015-01-02 ... 2100-12-31 
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+
<bound method Mapping.values of <xarray.Dataset> Size: 54MB
Dimensions:  (lat: 12, lon: 36, time: 31390)
Coordinates:
  * lat      (lat) float64 96B 40.2 40.45 40.7 40.95 ... 42.2 42.45 42.7 42.95
  * lon      (lon) float64 288B 256.1 256.3 256.6 256.8 ... 264.3 264.6 264.8
  * time     (time) datetime64[ns] 251kB 2015-01-01 2015-01-02 ... 2100-12-31
Data variables:
    tasmax   (time, lat, lon) float32 54MB ...>
vardata/cmip6/ssp585/tasmax/Downscaled_CMIP6_tasmax_Nebraska_ssp585_INM-CM4-8.nc closed successfully!
Coordinates:
  * lat  